In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd

# Replace 'your_folder/your_file.csv' with the actual path to your file in Google Drive
# For example: '/content/drive/My Drive/Colab Notebooks/my_data.csv'
file_path = '/content/drive/My Drive/BloomDataset/Data2.csv'

try:
    df = pd.read_csv(file_path)
    print(f"Successfully loaded data from {file_path}")
    display(df.head())
except FileNotFoundError:
    print(f"Error: The file '{file_path}' was not found. Please ensure the path is correct and the file exists in your mounted Google Drive.")
except Exception as e:
    print(f"An error occurred: {e}")

Successfully loaded data from /content/drive/My Drive/BloomDataset/Data2.csv


,Question,Label
0,Can you list the ingredients needed for a clas...,Remember
1,What are the top five tourist attractions in P...,Remember
2,List five common programming languages.,Remember
3,Name three types of renewable energy sources.,Remember
4,Can you list the planets in our solar system?,Remember


In [3]:
print("Installing NLTK...")
!pip install nltk
print("NLTK installed successfully.")

Installing NLTK...
NLTK installed successfully.


In [4]:
import nltk
nltk.download('stopwords')
nltk.download('punkt')
print("NLTK stopwords and punkt resources downloaded.")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...


NLTK stopwords and punkt resources downloaded.


[nltk_data]   Unzipping tokenizers/punkt.zip.


In [5]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import nltk

nltk.download('punkt_tab', quiet=True) # Download punkt_tab to resolve the LookupError

# Get English stop words
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    # Tokenize the text
    tokens = word_tokenize(text.lower())  # Convert to lowercase and tokenize
    # Remove stop words and non-alphabetic tokens
    processed_tokens = [word for word in tokens if word.isalpha() and word not in stop_words]
    return processed_tokens

# Apply the function to the 'Question' column
df['Processed_Questions'] = df['Question'].apply(preprocess_text)

print("Stop word reduction and tokenization applied to 'Question' column.")
display(df[['Question', 'Processed_Questions']].head())

Stop word reduction and tokenization applied to 'Question' column.


,Question,Processed_Questions
0,Can you list the ingredients needed for a clas...,"[list, ingredients, needed, classic, spaghetti..."
1,What are the top five tourist attractions in P...,"[top, five, tourist, attractions, paris]"
2,List five common programming languages.,"[list, five, common, programming, languages]"
3,Name three types of renewable energy sources.,"[name, three, types, renewable, energy, sources]"
4,Can you list the planets in our solar system?,"[list, planets, solar, system]"


In [6]:
df['Processed_Questions_Str'] = df['Processed_Questions'].apply(lambda x: ' '.join(x))

print("Converted 'Processed_Questions' list of tokens to a single string in 'Processed_Questions_Str' column.")
display(df[['Question', 'Processed_Questions', 'Processed_Questions_Str']].head())

Converted 'Processed_Questions' list of tokens to a single string in 'Processed_Questions_Str' column.


,Question,Processed_Questions,Processed_Questions_Str
0,Can you list the ingredients needed for a clas...,"[list, ingredients, needed, classic, spaghetti...",list ingredients needed classic spaghetti recipe
1,What are the top five tourist attractions in P...,"[top, five, tourist, attractions, paris]",top five tourist attractions paris
2,List five common programming languages.,"[list, five, common, programming, languages]",list five common programming languages
3,Name three types of renewable energy sources.,"[name, three, types, renewable, energy, sources]",name three types renewable energy sources
4,Can you list the planets in our solar system?,"[list, planets, solar, system]",list planets solar system


# Task
Convert the 'Processed_Questions_Str' column to numerical features using TF-IDF vectorization and encode the 'Label' column into numerical format. Then, initialize a Support Vector Machine (SVM) classifier and set up a Stratified K-Fold cross-validation strategy, along with SMOTE for class balancing. Subsequently, apply SMOTE to the training data in each cross-validation fold, train the SVM model, and evaluate its performance. Finally, summarize the cross-validation results by calculating and displaying the average performance metrics across all folds to assess the model's effectiveness with SMOTE.

## Feature Extraction and Label Encoding

### Subtask:
Convert the 'Processed_Questions_Str' column into numerical features using TF-IDF vectorization. Also, encode the 'Label' column into numerical format for model training.


**Reasoning**:
I need to import TfidfVectorizer and LabelEncoder from scikit-learn to perform feature extraction and label encoding as per the instructions.



In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder

print("TfidfVectorizer and LabelEncoder imported successfully.")

TfidfVectorizer and LabelEncoder imported successfully.


**Reasoning**:
Now that the necessary modules are imported, I will initialize the `TfidfVectorizer` and `LabelEncoder`, then apply them to the respective columns to convert text to numerical features and labels.



In [8]:
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(df['Processed_Questions_Str'])

encoder = LabelEncoder()
y = encoder.fit_transform(df['Label'])

print("TF-IDF features (X) and encoded labels (y) created successfully.")
print(f"Shape of TF-IDF features (X): {X.shape}")
print(f"Shape of encoded labels (y): {y.shape}")


TF-IDF features (X) and encoded labels (y) created successfully.
Shape of TF-IDF features (X): (1771, 1994)
Shape of encoded labels (y): (1771,)


## Initialize SVM and Cross-Validation

### Subtask:
Initialize a Support Vector Machine (SVM) classifier and set up a Stratified K-Fold cross-validation strategy to maintain class proportions across folds. Initialize SMOTE for balancing the classes.


**Reasoning**:
I need to import the required libraries and then initialize the Support Vector Machine (SVC) classifier, Stratified K-Fold cross-validation, and SMOTE for class balancing as per the instructions.



In [9]:
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold
from imblearn.over_sampling import SMOTE

# Initialize the SVM classifier
svm_classifier = SVC(random_state=42)
print("SVM classifier initialized.")

# Initialize Stratified K-Fold cross-validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print("Stratified K-Fold cross-validation initialized.")

# Initialize SMOTE for class balancing
smote = SMOTE(random_state=42)
print("SMOTE initialized.")

SVM classifier initialized.
Stratified K-Fold cross-validation initialized.
SMOTE initialized.


## Apply SMOTE, Train SVM, and Evaluate

### Subtask:
Implement a loop for K-Fold cross-validation. In each fold, apply SMOTE to the training data to balance classes, train the SVM model on the balanced training data, and evaluate its performance (e.g., accuracy, precision, recall, F1-score) on the corresponding test data. Store the evaluation metrics for each fold.


**Reasoning**:
To implement the cross-validation loop as instructed, I need to import the required evaluation metrics from `sklearn.metrics`, initialize lists to store the results from each fold, and then write the loop that iterates through the folds, applies SMOTE, trains the SVM model, makes predictions, and calculates and stores the performance metrics.



In [10]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# Initialize lists to store metrics for each fold
accuracy_scores = []
precision_scores = []
recall_scores = []
f1_scores = []

# Convert X to a dense array for easier indexing if it's a sparse matrix
# This step might be necessary for SMOTE or SVM if they don't handle sparse matrices directly
# However, `fit_resample` and `fit` methods for SMOTE and SVC generally handle sparse matrices.
# We will proceed assuming they can handle sparse matrices from TfidfVectorizer.

for fold, (train_index, test_index) in enumerate(skf.split(X, y)):
    print(f"\n--- Fold {fold + 1}/{skf.get_n_splits()} ---")

    # Split data into training and testing sets for the current fold
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    # Apply SMOTE to the training data
    print("Applying SMOTE to training data...")
    X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)
    print(f"Original training shape: {X_train.shape}, Resampled training shape: {X_train_resampled.shape}")

    # Train the SVM classifier on the resampled training data
    print("Training SVM classifier...")
    svm_classifier.fit(X_train_resampled, y_train_resampled)
    print("SVM classifier trained.")

    # Make predictions on the test data
    print("Making predictions on test data...")
    y_pred = svm_classifier.predict(X_test)

    # Calculate and store performance metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    recall = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)

    accuracy_scores.append(accuracy)
    precision_scores.append(precision)
    recall_scores.append(recall)
    f1_scores.append(f1)

    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1-Score: {f1:.4f}")
    # Optional: print confusion matrix for detailed analysis
    # print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

print("Cross-validation complete.")


--- Fold 1/5 ---
Applying SMOTE to training data...
Original training shape: (1416, 1994), Resampled training shape: (1440, 1994)
Training SVM classifier...
SVM classifier trained.
Making predictions on test data...
Accuracy: 0.9493
Precision: 0.9498
Recall: 0.9493
F1-Score: 0.9493

--- Fold 2/5 ---
Applying SMOTE to training data...
Original training shape: (1417, 1994), Resampled training shape: (1440, 1994)
Training SVM classifier...
SVM classifier trained.
Making predictions on test data...
Accuracy: 0.9435
Precision: 0.9458
Recall: 0.9435
F1-Score: 0.9436

--- Fold 3/5 ---
Applying SMOTE to training data...
Original training shape: (1417, 1994), Resampled training shape: (1440, 1994)
Training SVM classifier...
SVM classifier trained.
Making predictions on test data...
Accuracy: 0.9407
Precision: 0.9431
Recall: 0.9407
F1-Score: 0.9408

--- Fold 4/5 ---
Applying SMOTE to training data...
Original training shape: (1417, 1994), Resampled training shape: (1440, 1994)
Training SVM clas

**Reasoning**:
Now that the cross-validation loop has completed and the performance metrics for each fold have been stored, the next step is to summarize these results by calculating and displaying the average of each metric (accuracy, precision, recall, and F1-score) across all folds. This will provide an overall assessment of the model's effectiveness.



In [11]:
import numpy as np

# Calculate average scores across all folds
mean_accuracy = np.mean(accuracy_scores)
mean_precision = np.mean(precision_scores)
mean_recall = np.mean(recall_scores)
mean_f1 = np.mean(f1_scores)

print("\n--- Average Cross-Validation Results ---")
print(f"Average Accuracy: {mean_accuracy:.4f}")
print(f"Average Precision: {mean_precision:.4f}")
print(f"Average Recall: {mean_recall:.4f}")
print(f"Average F1-Score: {mean_f1:.4f}")


--- Average Cross-Validation Results ---
Average Accuracy: 0.9373
Average Precision: 0.9392
Average Recall: 0.9373
Average F1-Score: 0.9375


## Final Task

### Subtask:
Present the final summary of the SVM model's performance with SMOTE and cross-validation, including insights into its effectiveness for the given task.


## Summary:

### Data Analysis Key Findings

*   **Feature Extraction and Label Encoding**: The 'Processed\_Questions\_Str' column was successfully transformed into numerical TF-IDF features, resulting in a matrix `X` with a shape of (1771, 1994). The 'Label' column was encoded into numerical labels `y` with a shape of (1771,).
*   **Model and Cross-Validation Setup**: A Support Vector Machine (SVM) classifier, a Stratified K-Fold cross-validation strategy with 5 splits, and SMOTE for class balancing were successfully initialized.
*   **Cross-Validation with SMOTE and SVM Performance**: The model was evaluated using 5-fold stratified cross-validation. In each fold, SMOTE was applied to balance the training data (e.g., original training shape (1416, 1994) resampled to (1440, 1994)), and the SVM classifier was trained and tested.
*   **Average Performance Metrics**: The SVM model, trained with SMOTE and evaluated using stratified k-fold cross-validation, demonstrated strong performance:
    *   Average Accuracy: 0.9373
    *   Average Precision: 0.9392
    *   Average Recall: 0.9373
    *   Average F1-Score: 0.9375

### Insights or Next Steps

*   The SVM model, combined with TF-IDF features and SMOTE for handling class imbalance, achieved high and consistent performance across all evaluation metrics, indicating its strong effectiveness for the given classification task.
*   Further hyperparameter tuning for the SVM classifier (e.g., C, kernel) and SMOTE (e.g., k\_neighbors) could potentially lead to marginal improvements in performance. Additionally, exploring alternative text vectorization techniques or advanced ensemble methods might offer further gains.


# Task
```llm_code
print("The SVM model's performance with SMOTE and 10-fold cross-validation has been summarized, including insights into its effectiveness, reasoning behind hyperparameter and kernel selection, and observations from confusion matrix analysis.")
```

## Feature Extraction and Label Encoding

### Subtask:
The 'Processed_Questions_Str' column was previously transformed into numerical TF-IDF features (X) and the 'Label' column encoded into numerical labels (y). This step is already completed, and the resulting features and labels will be reused.


## Configure SVM, 10-Fold Cross-Validation, and SMOTE

### Subtask:
Re-initialize the Support Vector Machine (SVM) classifier with chosen hyperparameters and a specified kernel (e.g., 'rbf', along with C and gamma values). Set up a Stratified K-Fold cross-validation strategy with 10 splits to maintain class proportions. Re-initialize SMOTE for balancing classes.


**Reasoning**:
I need to initialize the SVM classifier with specified hyperparameters, set up a Stratified K-Fold cross-validation with 10 splits, and initialize SMOTE for class balancing as per the instructions.



In [12]:
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold
from imblearn.over_sampling import SMOTE

# Re-initialize the SVM classifier with chosen hyperparameters
svm_classifier = SVC(kernel='rbf', C=100, gamma='scale', random_state=42)
print("SVM classifier re-initialized with kernel='rbf', C=100, gamma='scale'.")

# Re-initialize Stratified K-Fold cross-validation with 10 splits
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
print("Stratified K-Fold cross-validation re-initialized with 10 splits.")

# Re-initialize SMOTE for class balancing
smote = SMOTE(random_state=42)
print("SMOTE re-initialized.")

SVM classifier re-initialized with kernel='rbf', C=100, gamma='scale'.
Stratified K-Fold cross-validation re-initialized with 10 splits.
SMOTE re-initialized.


## Perform 10-Fold Cross-Validation with SMOTE, Train SVM, and Evaluate

### Subtask:
Implement a loop for 10-Fold cross-validation. In each fold, apply SMOTE to the training data to balance classes, train the configured SVM model on the balanced training data, and evaluate its performance. Collect and store evaluation metrics (accuracy, precision, recall, F1-score) and the confusion matrix for each fold.


**Reasoning**:
To implement the 10-fold cross-validation loop as instructed, I will adapt the previously used structure, incorporating the re-initialized SVM classifier and 10-fold cross-validation object. I need to initialize a list for confusion matrices and then update the loop to calculate and store all specified performance metrics, including the confusion matrix, for each fold.



In [13]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import numpy as np

# Initialize lists to store metrics for each fold
accuracy_scores = []
precision_scores = []
recall_scores = []
f1_scores = []
confusion_matrices = [] # New list to store confusion matrices

for fold, (train_index, test_index) in enumerate(skf.split(X, y)):
    print(f"\n--- Fold {fold + 1}/{skf.get_n_splits()} ---")

    # Split data into training and testing sets for the current fold
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    # Apply SMOTE to the training data
    print("Applying SMOTE to training data...")
    X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)
    print(f"Original training shape: {X_train.shape}, Resampled training shape: {X_train_resampled.shape}")

    # Train the SVM classifier on the resampled training data
    print("Training SVM classifier...")
    svm_classifier.fit(X_train_resampled, y_train_resampled)
    print("SVM classifier trained.")

    # Make predictions on the test data
    print("Making predictions on test data...")
    y_pred = svm_classifier.predict(X_test)

    # Calculate and store performance metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    recall = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
    cm = confusion_matrix(y_test, y_pred)

    accuracy_scores.append(accuracy)
    precision_scores.append(precision)
    recall_scores.append(recall)
    f1_scores.append(f1)
    confusion_matrices.append(cm)

    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1-Score: {f1:.4f}")
    print("Confusion Matrix:\n", cm)

print("Cross-validation complete.")


--- Fold 1/10 ---
Applying SMOTE to training data...
Original training shape: (1593, 1994), Resampled training shape: (1620, 1994)
Training SVM classifier...
SVM classifier trained.
Making predictions on test data...
Accuracy: 0.9719
Precision: 0.9724
Recall: 0.9719
F1-Score: 0.9714
Confusion Matrix:
 [[30  0  0  0  0  0]
 [ 0 30  0  0  0  0]
 [ 0  0 29  0  0  1]
 [ 0  0  0 30  0  0]
 [ 0  0  0  0 28  0]
 [ 1  0  0  2  1 26]]

--- Fold 2/10 ---
Applying SMOTE to training data...
Original training shape: (1594, 1994), Resampled training shape: (1620, 1994)
Training SVM classifier...
SVM classifier trained.
Making predictions on test data...
Accuracy: 0.9435
Precision: 0.9453
Recall: 0.9435
F1-Score: 0.9437
Confusion Matrix:
 [[28  1  0  0  0  1]
 [ 0 27  2  0  1  0]
 [ 0  0 29  1  0  0]
 [ 0  0  1 28  0  1]
 [ 0  1  0  0 26  0]
 [ 0  0  1  0  0 29]]

--- Fold 3/10 ---
Applying SMOTE to training data...
Original training shape: (1594, 1994), Resampled training shape: (1620, 1994)
Traini

**Reasoning**:
The 10-fold cross-validation loop has completed, and all metrics (accuracy, precision, recall, F1-score, and confusion matrices) have been collected and stored in their respective lists. The next logical step is to summarize these results by calculating and displaying the average of each metric across all folds, which provides an overall assessment of the model's effectiveness with the new hyperparameters and 10-fold cross-validation.



In [14]:
import numpy as np

# Calculate average scores across all folds
mean_accuracy = np.mean(accuracy_scores)
mean_precision = np.mean(precision_scores)
mean_recall = np.mean(recall_scores)
mean_f1 = np.mean(f1_scores)

print("\n--- Average 10-Fold Cross-Validation Results ---")
print(f"Average Accuracy: {mean_accuracy:.4f}")
print(f"Average Precision: {mean_precision:.4f}")
print(f"Average Recall: {mean_recall:.4f}")
print(f"Average F1-Score: {mean_f1:.4f}")


--- Average 10-Fold Cross-Validation Results ---
Average Accuracy: 0.9497
Average Precision: 0.9507
Average Recall: 0.9497
Average F1-Score: 0.9495


## Analyze and Summarize Results

### Subtask:
Calculate and display the average performance metrics (accuracy, precision, recall, F1-score) across all 10 folds. Provide a clear explanation of each evaluation metric and its implications in the context of the classification task. Analyze the collected confusion matrices to understand the model's classification patterns and discuss the justification for the chosen hyperparameters and kernel.


### Explanation of Evaluation Metrics

*   **Accuracy**: Accuracy measures the proportion of total predictions that were correct. It is calculated as (True Positives + True Negatives) / (True Positives + True Negatives + False Positives + False Negatives). While intuitive, accuracy can be misleading in cases of imbalanced datasets, where a model might achieve high accuracy by simply predicting the majority class.

*   **Precision**: Precision measures the proportion of positive identifications that were actually correct. It is calculated as True Positives / (True Positives + False Positives). High precision relates to a low false positive rate, meaning that when the model predicts a positive outcome, it is usually correct. It is crucial when the cost of a false positive is high.

*   **Recall (Sensitivity)**: Recall measures the proportion of actual positives that were identified correctly. It is calculated as True Positives / (True Positives + False Negatives). High recall relates to a low false negative rate, meaning that the model is good at finding all positive samples. It is crucial when the cost of a false negative is high.

*   **F1-Score**: The F1-score is the harmonic mean of Precision and Recall. It provides a single metric that balances both precision and recall. It is particularly useful when the class distribution is uneven, as it penalizes models that favor one metric over the other. The F1-score is calculated as 2 * (Precision * Recall) / (Precision + Recall).

**Reasoning**:
To get an overall view of the model's classification patterns across all folds, I will calculate the sum of all individual confusion matrices. This will provide a consolidated confusion matrix that shows total true positives, false positives, true negatives, and false negatives for each class.



In [15]:
aggregated_cm = np.sum(confusion_matrices, axis=0)
print("\n--- Aggregated Confusion Matrix Across All Folds ---")
print(aggregated_cm)


--- Aggregated Confusion Matrix Across All Folds ---
[[285   4   0   4   3   4]
 [  1 287   6   1   3   2]
 [  0   3 291   3   2   1]
 [  0   0   2 292   2   4]
 [  4   5   1   2 249  10]
 [  6   1   1   6   8 278]]


### Analysis of Aggregated Confusion Matrix and Hyperparameter Justification

#### Aggregated Confusion Matrix Analysis:
The aggregated confusion matrix across all 10 folds is:
```
[[285   4   0   4   3   4]
 [  1 287   6   1   3   2]
 [  0   3 291   3   2   1]
 [  0   0   2 292   2   4]
 [  4   5   1   2 249  10]
 [  6   1   1   6   8 278]]
```
Each row represents the true class, and each column represents the predicted class.

From the matrix, we can observe the following:
*   **Class 0 (Row 0):** Mostly correctly classified (285). Some misclassifications with Class 1 (4), Class 3 (4), Class 4 (3), and Class 5 (4).
*   **Class 1 (Row 1):** Mostly correctly classified (287). Notable misclassifications with Class 2 (6), Class 4 (3), and Class 5 (2).
*   **Class 2 (Row 2):** Strong performance (291 correct). Minor confusion with Class 1 (3), Class 3 (3), Class 4 (2), and Class 5 (1).
*   **Class 3 (Row 3):** Excellent performance (292 correct). Minor confusion with Class 2 (2), Class 4 (2), and Class 5 (4).
*   **Class 4 (Row 4):** Generally well-classified (249 correct), but shows the most significant misclassifications, particularly with Class 5 (10 false negatives) and some with Class 0 (4), Class 1 (5), Class 2 (1), and Class 3 (2).
*   **Class 5 (Row 5):** Well-classified (278 correct), but frequently confused with Class 0 (6), Class 3 (6), and Class 4 (8). Also minor confusion with Class 1 (1) and Class 2 (1).

**Implications of Misclassifications:**
*   The diagonal elements are high, indicating good overall classification performance across all classes, aligning with the high average accuracy, precision, recall, and F1-score.
*   Classes 4 and 5 seem to have more mutual confusion compared to other pairs (e.g., 10 instances of true Class 4 predicted as Class 5, and 8 instances of true Class 5 predicted as Class 4). This suggests that the features for these two classes might be more similar or overlapping, making it harder for the model to distinguish them. Further investigation into the characteristics of questions belonging to these classes could be beneficial.
*   Other misclassifications are relatively low, indicating that the model effectively differentiates most class pairs.

#### Justification for Chosen Hyperparameters and Kernel:

*   **`kernel='rbf'` (Radial Basis Function Kernel):** The RBF kernel is a popular choice for SVMs, especially when dealing with non-linearly separable data. Text data, transformed into TF-IDF features, is often complex and high-dimensional, and relationships between classes are rarely linear. An RBF kernel allows the SVM to map the input features into a higher-dimensional space where classes might become linearly separable, enabling it to capture complex patterns. Given the strong performance (average F1-score around 0.95), the RBF kernel was effective in handling the non-linear nature of the text features.

*   **`C=100` (Regularization Parameter):** The `C` parameter controls the trade-off between achieving a low training error and a low testing error (i.e., avoiding overfitting). A high `C` value (like 100) tells the SVM to minimize the misclassification of training points. This can lead to a more complex decision boundary that tries to classify every training point correctly. While a very high `C` can lead to overfitting, in this case, coupled with the Stratified K-Fold cross-validation and SMOTE, `C=100` appears to have struck a good balance. The high scores suggest that the model is powerful enough to learn the underlying patterns without excessively overfitting, potentially due to the robust cross-validation setup and the benefits of SMOTE.

*   **`gamma='scale'` (Kernel Coefficient):** The `gamma` parameter defines how much influence a single training example has. With `gamma='scale'`, the value is set to `1 / (n_features * X.var())`, which means it is inversely proportional to the number of features and the variance of the data. This automatic scaling helps in making the model more robust by preventing `gamma` from being too large (leading to overfitting) or too small (leading to underfitting) given the high dimensionality of the TF-IDF features. This adaptive approach likely contributed to the consistent and high performance across different folds.

### Analysis of Aggregated Confusion Matrix and Hyperparameter Justification

#### Aggregated Confusion Matrix Analysis:
The aggregated confusion matrix across all 10 folds is:
```
[[285   4   0   4   3   4]
 [  1 287   6   1   3   2]
 [  0   3 291   3   2   1]
 [  0   0   2 292   2   4]
 [  4   5   1   2 249  10]
 [  6   1   1   6   8 278]]
```
Each row represents the true class, and each column represents the predicted class.

From the matrix, we can observe the following:
*   **Class 0 (Row 0):** Mostly correctly classified (285). Some misclassifications with Class 1 (4), Class 3 (4), Class 4 (3), and Class 5 (4).
*   **Class 1 (Row 1):** Mostly correctly classified (287). Notable misclassifications with Class 2 (6), Class 4 (3), and Class 5 (2).
*   **Class 2 (Row 2):** Strong performance (291 correct). Minor confusion with Class 1 (3), Class 3 (3), Class 4 (2), and Class 5 (1).
*   **Class 3 (Row 3):** Excellent performance (292 correct). Minor confusion with Class 2 (2), Class 4 (2), and Class 5 (4).
*   **Class 4 (Row 4):** Generally well-classified (249 correct), but shows the most significant misclassifications, particularly with Class 5 (10 false negatives) and some with Class 0 (4), Class 1 (5), Class 2 (1), and Class 3 (2).
*   **Class 5 (Row 5):** Well-classified (278 correct), but frequently confused with Class 0 (6), Class 3 (6), and Class 4 (8). Also minor confusion with Class 1 (1) and Class 2 (1).

**Implications of Misclassifications:**
*   The diagonal elements are high, indicating good overall classification performance across all classes, aligning with the high average accuracy, precision, recall, and F1-score.
*   Classes 4 and 5 seem to have more mutual confusion compared to other pairs (e.g., 10 instances of true Class 4 predicted as Class 5, and 8 instances of true Class 5 predicted as Class 4). This suggests that the features for these two classes might be more similar or overlapping, making it harder for the model to distinguish them. Further investigation into the characteristics of questions belonging to these classes could be beneficial.
*   Other misclassifications are relatively low, indicating that the model effectively differentiates most class pairs.

#### Justification for Chosen Hyperparameters and Kernel:

*   **`kernel='rbf'` (Radial Basis Function Kernel):** The RBF kernel is a popular choice for SVMs, especially when dealing with non-linearly separable data. Text data, transformed into TF-IDF features, is often complex and high-dimensional, and relationships between classes are rarely linear. An RBF kernel allows the SVM to map the input features into a higher-dimensional space where classes might become linearly separable, enabling it to capture complex patterns. Given the strong performance (average F1-score around 0.95), the RBF kernel was effective in handling the non-linear nature of the text features.

*   **`C=100` (Regularization Parameter):** The `C` parameter controls the trade-off between achieving a low training error and a low testing error (i.e., avoiding overfitting). A high `C` value (like 100) tells the SVM to minimize the misclassification of training points. This can lead to a more complex decision boundary that tries to classify every training point correctly. While a very high `C` can lead to overfitting, in this case, coupled with the Stratified K-Fold cross-validation and SMOTE, `C=100` appears to have struck a good balance. The high scores suggest that the model is powerful enough to learn the underlying patterns without excessively overfitting, potentially due to the robust cross-validation setup and the benefits of SMOTE.

*   **`gamma='scale'` (Kernel Coefficient):** The `gamma` parameter defines how much influence a single training example has. With `gamma='scale'`, the value is set to `1 / (n_features * X.var())`, which means it is inversely proportional to the number of features and the variance of the data. This automatic scaling helps in making the model more robust by preventing `gamma` from being too large (leading to overfitting) or too small (leading to underfitting) given the high dimensionality of the TF-IDF features. This adaptive approach likely contributed to the consistent and high performance across different folds.

## Final Task

### Subtask:
Present a comprehensive summary of the SVM model's performance with SMOTE and 10-fold cross-validation, including detailed insights into its effectiveness, the reasoning behind hyperparameter and kernel selection, and observations from the confusion matrix analysis.


## Summary:

### Data Analysis Key Findings

*   The SVM model, utilizing a Radial Basis Function (RBF) kernel with `C=100` and `gamma='scale'`, was evaluated using 10-fold stratified cross-validation on TF-IDF features, with SMOTE applied in each training fold to address class imbalance.
*   The model demonstrated strong overall performance across the 10 folds, achieving the following average metrics:
    *   **Average Accuracy**: 0.9497
    *   **Average Precision**: 0.9507
    *   **Average Recall**: 0.9497
    *   **Average F1-Score**: 0.9495
*   The **aggregated confusion matrix** across all folds showed high values along the diagonal, confirming the model's general effectiveness in correctly classifying instances.
*   The most notable misclassifications occurred between **Class 4** and **Class 5**, with 10 instances of true Class 4 being predicted as Class 5, and 8 instances of true Class 5 being predicted as Class 4. This suggests a higher degree of similarity or overlap in the features characterizing these two classes compared to others.
*   The selection of the `rbf` kernel was justified due to its ability to handle non-linear relationships prevalent in high-dimensional text data.
*   A high `C` value of 100 was chosen to minimize training error, which, in conjunction with cross-validation and SMOTE, helped the model learn complex patterns without significant overfitting.
*   The `gamma='scale'` parameter provided an adaptive kernel coefficient, contributing to the model's robustness and consistent performance across diverse data splits.

### Insights or Next Steps

*   Further investigation into the specific characteristics of the questions belonging to Class 4 and Class 5 could reveal why the model confuses them, potentially leading to more refined feature engineering or labeling.
*   Consider exploring advanced techniques for handling difficult-to-distinguish classes, such as ensemble methods focusing on these specific classes, or employing a more granular hyperparameter tuning strategy (e.g., GridSearchCV with a narrower range for C and gamma) to potentially improve the separation between Class 4 and Class 5.


# Task
The task to "Present a comprehensive summary of the SVM model's performance with the optimally tuned hyperparameters, SMOTE, and 10-fold cross-validation, including detailed insights into its effectiveness, the reasoning behind the chosen hyperparameters, and observations from the confusion matrix analysis." has been fully completed.

## Configure Grid Search for Hyperparameter Tuning

### Subtask:
Define a parameter grid with various C and gamma values to explore. Set up GridSearchCV to perform an exhaustive search over these parameters, using the StratifiedKFold (10 splits) and integrating SMOTE within the cross-validation process.


**Reasoning**:
To set up the GridSearchCV as instructed, I will first import the necessary modules, then define the parameter grid for the SVM hyperparameters. Following that, I will construct a pipeline that integrates SMOTE and the SVM classifier, and finally, initialize GridSearchCV with this pipeline, parameter grid, and the specified cross-validation strategy and scoring metric.



In [16]:
from sklearn.model_selection import GridSearchCV
from imblearn.pipeline import Pipeline

# Define the parameter grid for SVM
param_grid = {
    'svc__C': [0.1, 1, 10, 100],
    'svc__gamma': ['scale', 'auto', 0.1, 1]
}
print("Parameter grid for C and gamma defined.")

# Create a pipeline that includes SMOTE and the SVM classifier
pipeline = Pipeline([
    ('smote', smote), # Reuse the already initialized smote object
    ('svc', svm_classifier) # Reuse the already initialized svm_classifier object
])
print("Pipeline (SMOTE -> SVM) created.")

# Initialize GridSearchCV
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=skf, # Reuse the already initialized skf object (10 splits)
    scoring='f1_weighted',
    n_jobs=-1, # Use all available cores
    verbose=2 # Set verbose to 2 to see the progress
)
print("GridSearchCV initialized with the pipeline, parameter grid, and 10-fold StratifiedKFold cross-validation.")

Parameter grid for C and gamma defined.
Pipeline (SMOTE -> SVM) created.
GridSearchCV initialized with the pipeline, parameter grid, and 10-fold StratifiedKFold cross-validation.


## Execute Grid Search and Find Best Parameters

### Subtask:
Run the GridSearchCV to train and evaluate the SVM model with different hyperparameter combinations across the 10-fold cross-validation. Identify the best performing C and gamma values based on a chosen scoring metric (e.g., F1-score).


**Reasoning**:
I will execute the GridSearchCV by calling its `.fit()` method with `X` and `y` to perform the hyperparameter tuning, then print the best parameters and best score found.



In [17]:
print("Starting GridSearchCV...")
grid_search.fit(X, y)

print("GridSearchCV complete.")
print("Best parameters found: ", grid_search.best_params_)
print("Best F1-score (weighted) achieved: ", grid_search.best_score_)

Starting GridSearchCV...
Fitting 10 folds for each of 16 candidates, totalling 160 fits
GridSearchCV complete.
Best parameters found:  {'svc__C': 10, 'svc__gamma': 0.1}
Best F1-score (weighted) achieved:  0.9568956014198576


## Evaluate Model with Best Hyperparameters

### Subtask:
Once the best hyperparameters are identified, evaluate the SVM model using these optimal parameters within the 10-fold cross-validation framework, applying SMOTE in each fold. Collect performance metrics (accuracy, precision, recall, F1-score) and confusion matrices for each fold.


**Reasoning**:
I need to extract the best estimator from the `grid_search` object, initialize lists for storing performance metrics and confusion matrices, and then iterate through the cross-validation folds to fit the best pipeline, make predictions, and collect the evaluation metrics as per the instructions.



In [18]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import numpy as np

# 1. Extract the best estimator
best_svm_pipeline = grid_search.best_estimator_
print(f"Best SVM Pipeline extracted with parameters: {best_svm_pipeline.named_steps['svc'].get_params()}")

# 2. Initialize lists to store metrics for each fold
accuracy_scores_tuned = []
precision_scores_tuned = []
recall_scores_tuned = []
f1_scores_tuned = []
confusion_matrices_tuned = []

# 3. Loop through each fold of the skf (Stratified K-Fold) cross-validation
for fold, (train_index, test_index) in enumerate(skf.split(X, y)):
    print(f"\n--- Fold {fold + 1}/{skf.get_n_splits()} (Tuned Model) ---")

    # 4. Split data into training and testing sets for the current fold
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    # 5. Fit the best_svm_pipeline to the training data
    # The pipeline internally handles applying SMOTE and training the SVM
    print("Fitting best SVM pipeline to training data (SMOTE applied internally)... ")
    best_svm_pipeline.fit(X_train, y_train)
    print("Best SVM pipeline fitted.")

    # 6. Make predictions on the test data
    print("Making predictions on test data...")
    y_pred = best_svm_pipeline.predict(X_test)

    # 7. Calculate and store performance metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    recall = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)

    # 8. Calculate the confusion_matrix
    cm = confusion_matrix(y_test, y_pred)

    accuracy_scores_tuned.append(accuracy)
    precision_scores_tuned.append(precision)
    recall_scores_tuned.append(recall)
    f1_scores_tuned.append(f1)
    confusion_matrices_tuned.append(cm)

    # 10. Print the performance metrics and confusion matrix for each fold
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1-Score: {f1:.4f}")
    print("Confusion Matrix:\n", cm)

print("Cross-validation with tuned model complete.")

Best SVM Pipeline extracted with parameters: {'C': 10, 'break_ties': False, 'cache_size': 200, 'class_weight': None, 'coef0': 0.0, 'decision_function_shape': 'ovr', 'degree': 3, 'gamma': 0.1, 'kernel': 'rbf', 'max_iter': -1, 'probability': False, 'random_state': 42, 'shrinking': True, 'tol': 0.001, 'verbose': False}

--- Fold 1/10 (Tuned Model) ---
Fitting best SVM pipeline to training data (SMOTE applied internally)... 
Best SVM pipeline fitted.
Making predictions on test data...
Accuracy: 0.9719
Precision: 0.9735
Recall: 0.9719
F1-Score: 0.9715
Confusion Matrix:
 [[30  0  0  0  0  0]
 [ 0 30  0  0  0  0]
 [ 0  0 29  0  1  0]
 [ 0  0  0 30  0  0]
 [ 0  0  0  0 28  0]
 [ 1  0  0  2  1 26]]

--- Fold 2/10 (Tuned Model) ---
Fitting best SVM pipeline to training data (SMOTE applied internally)... 
Best SVM pipeline fitted.
Making predictions on test data...
Accuracy: 0.9718
Precision: 0.9734
Recall: 0.9718
F1-Score: 0.9717
Confusion Matrix:
 [[29  0  0  0  0  1]
 [ 0 28  1  0  1  0]
 [ 0 

**Reasoning**:
The 10-fold cross-validation with the tuned model has completed, and all performance metrics and confusion matrices for each fold have been collected. The next step is to calculate and display the average of these metrics across all folds to provide an overall assessment of the model's effectiveness with the optimized hyperparameters.



In [19]:
import numpy as np

# Calculate average scores across all folds for the tuned model
mean_accuracy_tuned = np.mean(accuracy_scores_tuned)
mean_precision_tuned = np.mean(precision_scores_tuned)
mean_recall_tuned = np.mean(recall_scores_tuned)
mean_f1_tuned = np.mean(f1_scores_tuned)

print("\n--- Average 10-Fold Cross-Validation Results (Tuned Model) ---")
print(f"Average Accuracy: {mean_accuracy_tuned:.4f}")
print(f"Average Precision: {mean_precision_tuned:.4f}")
print(f"Average Recall: {mean_recall_tuned:.4f}")
print(f"Average F1-Score: {mean_f1_tuned:.4f}")


--- Average 10-Fold Cross-Validation Results (Tuned Model) ---
Average Accuracy: 0.9571
Average Precision: 0.9580
Average Recall: 0.9571
Average F1-Score: 0.9569


**Reasoning**:
To analyze the model's classification patterns across all folds of the tuned model, I will calculate the sum of all individual confusion matrices. This will provide a consolidated confusion matrix that shows total true positives, false positives, true negatives, and false negatives for each class, which is a crucial part of the current subtask.



In [20]:
aggregated_cm_tuned = np.sum(confusion_matrices_tuned, axis=0)
print("\n--- Aggregated Confusion Matrix Across All Folds (Tuned Model) ---")
print(aggregated_cm_tuned)


--- Aggregated Confusion Matrix Across All Folds (Tuned Model) ---
[[291   3   0   2   0   4]
 [  1 291   5   0   2   1]
 [  0   3 295   0   2   0]
 [  2   0   2 289   1   6]
 [  1   4   1   1 253  11]
 [  7   2   0   7   8 276]]


### Analysis of Aggregated Confusion Matrix (Tuned Model) and Hyperparameter Justification

#### Aggregated Confusion Matrix Analysis (Tuned Model):
The aggregated confusion matrix for the model with tuned hyperparameters (C=10, gamma=0.1) across all 10 folds is:
```
[[291   3   0   2   0   4]
 [  1 291   5   0   2   1]
 [  0   3 295   0   2   0]
 [  2   0   2 289   1   6]
 [  1   4   1   1 253  11]
 [  7   2   0   7   8 276]]
```
Each row represents the true class, and each column represents the predicted class.

Comparing this to the initial model's aggregated confusion matrix, we observe a general improvement in correct classifications (higher diagonal values) and a reduction in misclassifications across several classes.

*   **Class 0 (Row 0):** Improved accuracy (291 correct vs. 285 in previous). Misclassifications with Class 1 (3), Class 3 (2), and Class 5 (4) are generally reduced or similar.
*   **Class 1 (Row 1):** Improved accuracy (291 correct vs. 287 in previous). Misclassifications with Class 2 (5) and Class 4 (2) show slight improvements.
*   **Class 2 (Row 2):** Maintained strong performance (295 correct vs. 291 in previous). Misclassifications remain low.
*   **Class 3 (Row 3):** Slightly reduced accuracy (289 correct vs. 292 in previous), but overall still very good. Misclassifications with Class 0 (2), Class 2 (2), Class 4 (1), and Class 5 (6) are present.
*   **Class 4 (Row 4):** Slightly improved accuracy (253 correct vs. 249 in previous). The misclassification with Class 5 remains the highest (11 false negatives), slightly increased from 10 in the previous model. This suggests that the confusion between Class 4 and Class 5 is a persistent challenge even with tuning.
*   **Class 5 (Row 5):** Slightly reduced accuracy (276 correct vs. 278 in previous). Misclassifications with Class 0 (7), Class 3 (7), and Class 4 (8) are still notable.

**Implications of Misclassifications (Tuned Model):**
*   The diagonal elements are largely high, confirming the overall good classification performance. The average F1-score improved from 0.9495 to 0.9569, indicating that the tuning successfully enhanced the model's ability to balance precision and recall.
*   The persistent and slightly increased confusion between **Class 4** and **Class 5** (11 instances of true Class 4 predicted as Class 5, and 8 instances of true Class 5 predicted as Class 4) highlights that these two classes are intrinsically difficult for the model to distinguish. This might be due to semantic overlap in the questions belonging to these categories, requiring potentially more advanced feature engineering or a more specialized model architecture.
*   The tuning has generally refined the decision boundaries, leading to better separation for most other classes.

#### Justification for Chosen Hyperparameters (Tuned Model):

After Grid Search, the optimal hyperparameters found were `C=10` and `gamma=0.1`.

*   **`C=10` (Regularization Parameter):** The tuning process reduced the optimal `C` value from the initial arbitrary `C=100` to `C=10`. This suggests that the initial `C=100` might have been too high, potentially leading to a more complex decision boundary that captured noise in the training data, even with cross-validation and SMOTE. A `C=10` indicates a slightly softer margin, allowing for a few more training errors in exchange for a simpler, more generalized decision boundary, which is beneficial for avoiding overfitting and improving performance on unseen data. The improved average F1-score (from 0.9495 to 0.9569) supports that `C=10` achieved a better balance between bias and variance.

*   **`gamma=0.1` (Kernel Coefficient):** The optimal `gamma` value was found to be `0.1`, moving from the 'scale' heuristic. A specific, smaller `gamma` value means that a single training example has a larger radius of influence. Compared to `gamma='scale'` which is data-dependent, `gamma=0.1` was empirically determined to be more effective. This specific value likely provided the right balance for the RBF kernel to capture the intricacies of the TF-IDF feature space without making the model too sensitive to individual data points (which a very high gamma would do) or too rigid (which a very low gamma would do). The improved performance metrics confirm that `gamma=0.1` allows the RBF kernel to effectively map the non-linear relationships in the data.

## Final Task

### Subtask:
Present a comprehensive summary of the SVM model's performance with the optimally tuned hyperparameters, SMOTE, and 10-fold cross-validation, including detailed insights into its effectiveness, the reasoning behind the chosen hyperparameters, and observations from the confusion matrix analysis.


## Summary:

### Q&A

*   **How did the SVM model perform with optimal hyperparameters, SMOTE, and 10-fold cross-validation?**
    The SVM model, incorporating SMOTE and optimally tuned hyperparameters (C=10, gamma=0.1), achieved strong performance across 10-fold cross-validation. The average weighted F1-score was approximately 0.9569, with an average accuracy of 0.9571, precision of 0.9580, and recall of 0.9571. This indicates a well-balanced model with high predictive power across classes.

*   **What was the reasoning behind the chosen hyperparameters?**
    The optimal hyperparameters were identified as `C=10` and `gamma=0.1` through Grid Search.
    *   `C=10`: This value was chosen over a higher initial value (like C=100) because it strikes a better balance between allowing some training errors and creating a more generalized decision boundary, thus preventing overfitting. A `C=10` provides a "softer margin," which improved the model's ability to generalize to unseen data, as evidenced by the increased average F1-score from 0.9495 to 0.9569.
    *   `gamma=0.1`: This specific gamma value was found to be more effective than the 'scale' heuristic. It allows the RBF kernel to capture intricate non-linear relationships within the TF-IDF feature space without making the model excessively sensitive to individual data points (which a very high gamma would do) or overly rigid (which a very low gamma would do).

*   **What observations were made from the confusion matrix analysis?**
    The aggregated confusion matrix showed a general improvement in correct classifications (higher diagonal values) and a reduction in misclassifications compared to a previous model. While overall performance was very good, a persistent and slightly increased confusion was observed between Class 4 and Class 5. Specifically, 11 instances of true Class 4 were predicted as Class 5, and 8 instances of true Class 5 were predicted as Class 4. This suggests that these two classes are intrinsically difficult for the model to distinguish, potentially due to semantic overlap in their content.

### Data Analysis Key Findings

*   The optimal hyperparameters identified through GridSearchCV were `C=10` and `gamma=0.1`.
*   The best F1-score (weighted) achieved during the Grid Search was approximately 0.9569.
*   The average performance metrics for the tuned SVM model with SMOTE across 10-fold cross-validation were:
    *   Average Accuracy: 0.9571
    *   Average Precision: 0.9580
    *   Average Recall: 0.9571
    *   Average F1-Score: 0.9569
*   The aggregated confusion matrix revealed generally high diagonal values, indicating strong performance. However, there was notable and persistent confusion between Class 4 and Class 5, with 11 misclassifications from true Class 4 to Class 5 and 8 from true Class 5 to Class 4.

### Insights or Next Steps

*   The tuning process successfully optimized the SVM model, leading to improved generalization and higher F1-scores, demonstrating the benefit of hyperparameter optimization combined with SMOTE for imbalanced datasets.
*   Further investigation is warranted for distinguishing between Class 4 and Class 5, potentially involving more advanced feature engineering, exploring different model architectures, or specialized techniques for these specific challenging classes.
